# Full-image grouped threshold calibration

This notebook calibrates one task-aware `prediction_threshold` per model on reconstructed full images. Semantic models use cached foreground-probability maps after SAHI stitching; instance models use cached scored instances. It is cache-only unless an explicit rerun switch is enabled. AOIs are kept intact across cross-validation folds.

In [ ]:
%pip install -q git+https://github.com/mooch443/dataset-fixer.git 'numpy<2.1,>=1.22'

In [ ]:
IN_COLAB = True

MODEL_SOURCES = [
    {
        "name": "yolo26x-sem-512px-sweep-best-thr0p60",
        "source": "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/islands-128-08.08.2026-merged-1class_masks-yolo26x-sem-512px-checkpoint-sweep-2026-08-11_11-34_20260811_113554",
        "run_file": "islands-128-08.08.2026-merged-1class_masks-yolo26x-sem-512px-checkpoint-sweep-2026-08-11_11-34-best-semantic-8666599f5405.zip",
        "task": "semantic",
        "native_tile_size": 128,
        "upscale_factor": 4,
        "prediction_threshold": 0.60,
    },
    {
        "name": "yolo26x-seg-256px-sweep-best-thr0p10",
        "source": "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/islands-128-08.08.2026-merged-1class_yolo-yolo26x-seg-256px-checkpoint-sweep-2026-08-11_11-26_20260811_112946",
        "run_file": "islands-128-08.08.2026-merged-1class_yolo-yolo26x-seg-256px-checkpoint-sweep-2026-08-11_11-26-best-segment-2e4461b86467.zip",
        "task": "segment",
        "native_tile_size": 128,
        "upscale_factor": 2,
        "prediction_threshold": 0.10,
    },
    {
        "name": "yolo26x-seg-256px-sweep-last-thr0p15",
        "source": "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/islands-128-08.08.2026-merged-1class_yolo-yolo26x-seg-256px-checkpoint-sweep-2026-08-11_11-26_20260811_112946",
        "run_file": "islands-128-08.08.2026-merged-1class_yolo-yolo26x-seg-256px-checkpoint-sweep-2026-08-11_11-26-last-segment-791d661199c3.zip",
        "task": "segment",
        "native_tile_size": 128,
        "upscale_factor": 2,
        "prediction_threshold": 0.15,
    },
    {
        "source": (
            "/content/drive/MyDrive/islands/08.08.2026-merged-1class_masks-"
            "yolo26x-sem-512px-2026-08-11_00-21_20260811_002334.pt"
        ),
        "task": "semantic",
        "native_tile_size": 128,
        "upscale_factor": 4,
    },
    {
        "source": "/content/drive/MyDrive/islands/islands-128-08.08.2026-merged-1class_yolo-128px-yolox.pt",
        "native_tile_size": 128,
        "upscale_factor": 2,
    },
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/8vdnx8hw",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/suc7mbqy",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/gnsuhtfc",
    {
        "source": "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/islands-128-08.08.2026-merged-1class_masks-yolo26m-sem-1024px-8x-2026-08-10_11-46_20260810_114819",
        "batch_size": 16,
    },
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/yur0cfln_20260810_145433",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/yolo26x-seg-256px-2x-2026-08-10_16-03_20260810_160602",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/koh81tx5",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/0ng82omc",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/i9xve33c",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/0we3e4aq",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/0g271i4w",
]

DATASET_SOURCE = (
    "/content/drive/MyDrive/islands/islands-fair-base-sem.zip"
    if IN_COLAB
    else "/Users/tristan/Downloads/island-dataset/islands-fair-base-sem"
)
CACHE_ARCHIVE = "/content/drive/MyDrive/islands/dataset-fixer-cache.zip"
CACHE_OVERLAY_DIRECTORY = "/content/drive/MyDrive/islands/dataset-fixer-cache-overlay"
DISCONNECT_AFTER_COMPLETION = True
SPLIT = "val"
INFERENCE = "sahi"
BATCH_SIZE = 96
WORKERS = 8
SAHI_OVERLAP = 0.15
THRESHOLDS = (0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90)
CV_FOLDS = 5
RERUN_MISSING_PROBABILITY_MAPS = True  # one-time population of reusable semantic score caches
RERUN_MISSING_INSTANCE_PREDICTIONS = False
CALIBRATION_DESTINATION = (
    "/content/drive/MyDrive/islands/full-image-threshold-calibration"
    if IN_COLAB
    else "/Users/tristan/Downloads/island-dataset/full-image-threshold-calibration"
)

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    archive = Path(CACHE_ARCHIVE)
    if archive.is_file() and not Path("/content/dataset-fixer-cache").exists():
        !rsync --progress {archive} /content/dataset-fixer-cache.zip
        !unzip -q /content/dataset-fixer-cache.zip -d /content
    cache_overlay = Path(CACHE_OVERLAY_DIRECTORY)
    if cache_overlay.is_dir():
        !rsync -a {cache_overlay}/ /content/dataset-fixer-cache/
    CACHE_SESSION_MARKER = Path("/content/dataset-fixer-cache-session-start")
    CACHE_SESSION_MARKER.touch()

# Required by the 184-epoch nnU-Net bundle in this cohort.
import nnunetv2
trainer_file = (
    Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" /
    "variants" / "training_length" / "nnUNetTrainer_184epochs.py"
)
trainer_file.parent.mkdir(parents=True, exist_ok=True)
if not trainer_file.is_file():
    trainer_file.write_text(
        "import torch\n"
        "from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n\n"
        "class nnUNetTrainer_184epochs(nnUNetTrainer):\n"
        "    def __init__(self, plans, configuration, fold, dataset_json, "
        "device=torch.device('cuda')):\n"
        "        super().__init__(plans, configuration, fold, dataset_json, device)\n"
        "        self.num_epochs = 184\n",
        encoding="utf-8",
    )

In [ ]:
import torch
import wandb
from dataset_fixer import Dataset, Model

wandb.login()
device = (
    "cuda" if torch.cuda.is_available() else
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else
    "cpu"
)
dataset = Dataset.open(DATASET_SOURCE)
models = Model.load_many(MODEL_SOURCES)
common = {
    "device": device,
    "workers": WORKERS,
    "inference": INFERENCE,
    "batch_size": BATCH_SIZE,
    "sahi_overlap": SAHI_OVERLAP,
    "nnunet_tta": False,
}
models = models.configure({name: common for name in models.names})
[(model.name, model.task, model.prediction_threshold) for model in models]

In [ ]:
import re

AOI_PATTERN = re.compile(
    r"(?i)([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-"
    r"[0-9a-f]{4}-[0-9a-f]{12})_20\d{6}(?:_|$)"
)

def get_aoi(filename: str | Path) -> str:
    match = AOI_PATTERN.search(Path(filename).name)
    if match is None:
        raise ValueError(f"Could not extract AOI UUID from {filename!r}")
    return match.group(1).lower()

In [ ]:
from dataset_fixer import calibrate_prediction_thresholds

calibration = calibrate_prediction_thresholds(
    models,
    dataset,
    split=SPLIT,
    group_by=get_aoi,
    thresholds=THRESHOLDS,
    folds=CV_FOLDS,
    prediction_cache=True,
    rerun_missing_probability_maps=RERUN_MISSING_PROBABILITY_MAPS,
    rerun_missing_instance_predictions=RERUN_MISSING_INSTANCE_PREDICTIONS,
    destination=CALIBRATION_DESTINATION,
    progress=True,
)
calibration.location

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(calibration.cache_audit))
display(pd.DataFrame(
    [
        {"model": name, "prediction_threshold": threshold}
        for name, threshold in calibration.recommendations.items()
    ]
))

In [ ]:
from copy import deepcopy
from pprint import pprint

CALIBRATED_MODEL_SOURCES = deepcopy(MODEL_SOURCES)
for index, model in enumerate(models):
    threshold = calibration.recommendations.get(model.name)
    if threshold is None:
        continue
    source = CALIBRATED_MODEL_SOURCES[index]
    if not isinstance(source, dict):
        source = {"source": source}
        CALIBRATED_MODEL_SOURCES[index] = source
    source.pop("confidence", None)
    source.pop("foreground_probability_threshold", None)
    source["prediction_threshold"] = round(float(threshold), 6)

pprint(CALIBRATED_MODEL_SOURCES, sort_dicts=False)

In [ ]:
if IN_COLAB:
    cache_overlay = Path(CACHE_OVERLAY_DIRECTORY)
    cache_overlay.mkdir(parents=True, exist_ok=True)
    # Persist only complete/new evaluation-cache files from this session.
    !cd /content/dataset-fixer-cache && find . -type f -path '*/.cache/evaluations/*' -newer {CACHE_SESSION_MARKER} -print0 | rsync -a --from0 --files-from=- ./ {cache_overlay}/
    print("Updated cache overlay:", cache_overlay)
    if DISCONNECT_AFTER_COMPLETION:
        from google.colab import runtime
        runtime.unassign()